# 07 · Resultados de los modelos para el dashboard
**Fase CRISP-DM:** 6. Despliegue

**Objetivo:** usar los modelos entrenados (notebooks 04 y 05) para generar las **predicciones de cada abulón** y guardarlas en un CSV. Ese archivo se carga en SQL Server y se muestra en **Power BI**.

Se marca cada registro como **Entrenamiento** o **Prueba**, usando exactamente la misma división (80/20, `random_state=42`) de los notebooks 04 y 05. En el dashboard se deben evaluar los modelos **solo con los datos de Prueba**, porque son los que el modelo nunca vio.

In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score

RUTA_PROC = Path("..") / "data" / "processed"
RUTA_MOD = Path("..") / "modelos"
SEMILLA = 42

df = pd.read_csv(RUTA_PROC / "abalone_limpio.csv")
X = df[["Length", "Diameter", "Height", "Whole_weight",
        "Shucked_weight", "Viscera_weight", "Shell_weight", "Sex"]]

modelo_reg = joblib.load(RUTA_MOD / "regresion_random_forest.joblib")
modelo_cla = joblib.load(RUTA_MOD / "clasificacion_random_forest.joblib")
print("Modelos cargados")

Modelos cargados


## 1. Identificar qué registros fueron de prueba
Se repite la misma división de los notebooks 04 (regresión) y 05 (clasificación, con `stratify`).

In [2]:
_, X_test_reg = train_test_split(X, test_size=0.2, random_state=SEMILLA)
_, X_test_cla = train_test_split(X, test_size=0.2, random_state=SEMILLA, stratify=df["Grupo_Edad"])

df["Conjunto_Regresion"] = np.where(df.index.isin(X_test_reg.index), "Prueba", "Entrenamiento")
df["Conjunto_Clasificacion"] = np.where(df.index.isin(X_test_cla.index), "Prueba", "Entrenamiento")
print(df["Conjunto_Regresion"].value_counts())

Conjunto_Regresion
Entrenamiento    3335
Prueba            834
Name: count, dtype: int64


## 2. Generar las predicciones

In [3]:
df["Rings_Predicho"] = modelo_reg.predict(X).round(2)
df["Edad_Predicha_anios"] = (df["Rings_Predicho"] + 1.5).round(2)
df["Error_Absoluto"] = (df["Rings"] - df["Rings_Predicho"]).abs().round(2)
df["Grupo_Predicho"] = modelo_cla.predict(X)
df["Acierto_Grupo"] = np.where(df["Grupo_Edad"] == df["Grupo_Predicho"], "Sí", "No")
df[["ID_Abulon", "Rings", "Rings_Predicho", "Error_Absoluto", "Grupo_Edad", "Grupo_Predicho", "Acierto_Grupo"]].head()

,ID_Abulon,Rings,Rings_Predicho,Error_Absoluto,Grupo_Edad,Grupo_Predicho,Acierto_Grupo
0,1,15,9.52,5.48,Maduro,Joven,No
1,2,7,7.95,0.95,Joven,Joven,Sí
2,3,9,10.74,1.74,Adulto,Adulto,Sí
3,4,10,9.69,0.31,Adulto,Adulto,Sí
4,5,7,6.68,0.32,Joven,Joven,Sí


## 3. Verificar que las métricas coinciden con los notebooks 04 y 05
Si coinciden, sabemos que el archivo para el dashboard es correcto.

In [4]:
p_reg = df[df["Conjunto_Regresion"] == "Prueba"]
p_cla = df[df["Conjunto_Clasificacion"] == "Prueba"]
print(f"Regresión (prueba)     -> MAE: {mean_absolute_error(p_reg['Rings'], p_reg['Rings_Predicho']):.3f} | "
      f"R²: {r2_score(p_reg['Rings'], p_reg['Rings_Predicho']):.3f}")
print(f"Clasificación (prueba) -> Exactitud: {accuracy_score(p_cla['Grupo_Edad'], p_cla['Grupo_Predicho']):.3f}")

Regresión (prueba)     -> MAE: 1.486 | R²: 0.612
Clasificación (prueba) -> Exactitud: 0.641


📌 Si calculas el error con **todos** los registros, saldrá más bajo, porque incluye los datos con los que el modelo **aprendió**. Por eso en el dashboard se filtra **Conjunto = Prueba** para mostrar el desempeño real.

## 4. Guardar

In [5]:
cols = ["ID_Abulon", "Conjunto_Regresion", "Conjunto_Clasificacion", "Rings_Predicho",
        "Edad_Predicha_anios", "Error_Absoluto", "Grupo_Predicho", "Acierto_Grupo"]
df[cols].to_csv(RUTA_PROC / "abalone_predicciones.csv", index=False, encoding="utf-8", lineterminator="\n")
print("Guardado: data/processed/abalone_predicciones.csv", df[cols].shape)

Guardado: data/processed/abalone_predicciones.csv (4169, 8)
